In [1]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from spectral.io import envi

In [3]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')
flights = '/store/carroll/col/2018/raw/rmbl/' # still only 4 / 7 days, will rerun once have everything

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'extracted_observations'

In [36]:
# load and view relevant schema and dtype for the table
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# fix typos
schema['column_name'] = schema['column_name'].replace('wavelenght_center', 'wavelength_center')

schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
4,extracted_observations,obs_type,USER-DEFINED
5,extracted_observations,obs_value,double precision
6,extracted_observations,obs_id,uuid
7,extracted_observations,pixel_id,uuid


In [9]:
# view relevant data-types info
# for now we can't do this automatically by column_name, because column_name is not explicitly linked to enum_type in any way yet

data_types = data_types[data_types['Enum Type']=='OBS_type']
data_types

,Schema,Enum Type,Enum Value
9,sbgplants,OBS_type,UTC time
10,sbgplants,OBS_type,aspect
11,sbgplants,OBS_type,cosine i
12,sbgplants,OBS_type,path length
13,sbgplants,OBS_type,slope
14,sbgplants,OBS_type,solar phase
15,sbgplants,OBS_type,to-sensor-azimuth
16,sbgplants,OBS_type,to-sensor-zenith
17,sbgplants,OBS_type,to-sun-azimuth
18,sbgplants,OBS_type,to-sun-zenith


In [21]:
len(data_types)

10

In [10]:
# load relevant output tables

pixel = pd.read_csv(os.path.join(out_folder, 'pixel.csv'))
flightlines = pixel.flightline_id.unique()

In [19]:
# fps_obs = [x for x in glob(os.path.join(flights, '*/*obs_ort.hdr')) if any(xx in x for xx in flightlines)]

# val_cols = envi.read_envi_header(fps_obs[0])['band names']

In [58]:
# extract rdn spectra

fps = [x for x in glob(os.path.join(flights, '*/*obs_ort.hdr')) if any(xx in x for xx in flightlines)]
id_cols = pixel.columns
val_cols = envi.read_envi_header(fps_obs[0])['band names']

out = []

for fp in fps:
    # filter px to flightline
    flight = fp.split('/')[-1].removesuffix('_rdn_obs_ort.hdr')
    tmp = pixel[pixel['flightline_id']==flight].copy()
    # extract obs
    obs = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = obs[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='obs_type', value_name='obs_value')
    out.append(tmp)

df = pd.concat(out)

In [60]:
# prepare & populate out table
out_table = df

# map obs_type to enumerated values
out_table = out_table[out_table.obs_type!='ATCOR to-sensor zenith']
obs_type = {
    'path length' : 'path length',
    'to-sensor azimuth' : 'to-sensor-azimuth',
    'to-sensor zenith' : 'to-sensor-zenith',
    'to-sun azimuth' : 'to-sun-azimuth',
    'to-sun zenith' : 'to-sun-zenith',
    'phase' : 'solar phase',
    'slope' : 'slope',
    'aspect' : 'aspect',
    'cosine i' : 'cosine i',
    'gps time' : 'UTC time'
}
out_table.loc[:,'obs_type'] = out_table['obs_type'].map(obs_type)

out_table['obs_id'] = pd.NA

# reorder columns
out_table = out_table[schema.column_name]

out_table

/tmp/ipykernel_170613/2741782077.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out_table['obs_id'] = pd.NA


,obs_type,obs_value,obs_id,pixel_id
0,path length,1683.907959,<NA>,px0
1,path length,1684.000244,<NA>,px1
2,path length,1683.652344,<NA>,px2
3,path length,1683.788208,<NA>,px3
4,to-sensor-azimuth,182.968033,<NA>,px0
...,...,...,...,...
1085,UTC time,17.564207,<NA>,px5471
1086,UTC time,17.564161,<NA>,px5472
1087,UTC time,17.564159,<NA>,px5473
1088,UTC time,17.564093,<NA>,px5474


In [62]:
# check, update final column data types

print(out_table.dtypes)

out_table.dtypes

obs_type      object
obs_value    float32
obs_id        object
pixel_id      object
dtype: object


obs_type      object
obs_value    float32
obs_id        object
pixel_id      object
dtype: object

In [63]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)